# NB00: Feature matrix assembly

Assembles the sample × feature matrix for hybrid metal prediction.

**Architecture**:
- 16S samples: `arkinlab.microbeatlas.sample_metadata` (sample_id, lat, lon)
- CWM features: `otu_counts_long` → genus RA → matrix multiply with density table
- Env covariates: `enriched_metadata_gee` (OLM pH, clay, water content, NDVI, temp, precip)
- Metal targets: spatial join of sample_metadata coords with `enriched_metadata` (GeoROC) within 50 km

**Outputs**
- `data/feature_matrix.parquet`
- `data/spatial_blocks.csv`
- `data/coverage_report.csv`

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path

# Add scripts/ to path; works whether CWD is notebooks/ or project root
for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(
    p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists()
)
DATA_DIR.mkdir(exist_ok=True)
print('DATA_DIR:', DATA_DIR)

In [ ]:
try:
    spark
except NameError:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
print('Spark ready:', spark.version)

## 1. Load soil sample IDs

In [ ]:
from env_utils import get_soil_sample_ids
sample_coords = get_soil_sample_ids(spark)
print(f'Soil samples with lat/lon: {len(sample_coords):,}')
sample_coords.head(3)

## 2. GEE environmental covariates

In [ ]:
from env_utils import get_gee_features
srs_keys = sample_coords['srs_key'].dropna().unique().tolist()
gee_features = get_gee_features(spark, srs_keys=srs_keys)
print(f'GEE features: {gee_features.shape}')
print('OLM pH (should be 2-12):', gee_features['ph_olm'].describe().round(2))

## 2.5. CSU metal mobility features

In [ ]:
from env_utils import get_csu_mobility_features, CSU_MOBILITY_COLS
csu_df = get_csu_mobility_features(spark, sample_coords)
mob_cols = list(CSU_MOBILITY_COLS.values())
n_matched = csu_df[mob_cols].dropna(how='all').shape[0]
print(f'CSU mobility features: {csu_df.shape}')
print(f'Samples with any CSU match: {n_matched:,} / {len(sample_coords):,}')
csu_df[mob_cols].describe().round(4)

## 3. Metal targets via spatial join

In [ ]:
from env_utils import get_metal_targets_by_spatial_join, SPATIAL_RADIUS_KM
print(f'Spatial join radius: {SPATIAL_RADIUS_KM} km')
metal_targets = get_metal_targets_by_spatial_join(spark, sample_coords, radius_km=SPATIAL_RADIUS_KM)
log_cols = [c for c in metal_targets.columns if c.startswith('log_')]
print(f'Samples with geochem match: {len(metal_targets):,}')
for c in log_cols:
    n = metal_targets[c].notna().sum()
    print(f'  {c}: {n:,}')
metal_targets[['n_geochem_pts'] + log_cols].describe().round(3)

## 4. CWM features

In [ ]:
from cwm_utils import load_otu_bridge, load_genus_densities, compute_cwm, load_genus_ra_from_spark

samples_with_targets = metal_targets.index[
    metal_targets[log_cols].notna().any(axis=1)
].tolist()
print(f'Samples to compute CWM for: {len(samples_with_targets):,}')

bridge = load_otu_bridge(DATA_DIR / 'otu_pangenome_link_v2.csv')
densities = load_genus_densities(genus_trait_path=DATA_DIR / 'genus_trait_table.csv')
print(f'Bridge: {len(bridge):,} OTUs | Densities: {len(densities):,} genera')

In [ ]:
genus_ra = load_genus_ra_from_spark(spark, sample_ids=samples_with_targets, otu_bridge=bridge)
print(f'Genus RA matrix: {genus_ra.shape}')

In [ ]:
cwm_df = compute_cwm(genus_ra, densities)
cwm_cols = [c for c in cwm_df.columns if c.startswith('CWM_')]
print(f'CWM matrix: {cwm_df.shape}')
print(f'Low coverage: {cwm_df["low_coverage_flag"].sum()} / {len(cwm_df)}')
cwm_df[cwm_cols].describe().round(4)

## 5. Assemble feature matrix

In [ ]:
from env_utils import build_feature_table, report_feature_coverage

feature_matrix = build_feature_table(
    sample_coords=sample_coords,
    gee_features=gee_features,
    metal_targets=metal_targets,
    cwm_df=cwm_df,
    csu_df=csu_df,
)

has_cwm = feature_matrix[cwm_cols].notna().any(axis=1)
log_target_cols = [c for c in feature_matrix.columns if c.startswith('log_') and '_ppm' in c]
has_target = feature_matrix[log_target_cols].notna().any(axis=1)
feature_matrix = feature_matrix[has_cwm & has_target].copy()
print(f'Final feature matrix: {feature_matrix.shape}')

In [ ]:
cov_report = report_feature_coverage(feature_matrix)
print('Lowest-coverage features:')
print(cov_report.head(15).to_string(index=False))

## 6. Spatial blocks

In [ ]:
from spatial_utils import make_spatial_blocks, report_block_sizes
block_labels = make_spatial_blocks(feature_matrix, n_blocks=5)
feature_matrix['block'] = block_labels
print(report_block_sizes(feature_matrix, block_labels).to_string(index=False))

## 7. Save

In [ ]:
feature_matrix.to_parquet(DATA_DIR / 'feature_matrix.parquet')
feature_matrix[['block']].reset_index().to_csv(DATA_DIR / 'spatial_blocks.csv', index=False)
cov_report.to_csv(DATA_DIR / 'coverage_report.csv', index=False)

print('=== NB00 COMPLETE ===')
print(f'  feature_matrix.parquet: {feature_matrix.shape}')
print(f'  pH source: {feature_matrix["ph_source"].value_counts().to_dict()}')
for c in log_target_cols:
    n = feature_matrix[c].notna().sum()
    print(f'  {c}: {n:,} samples')